**Atividade**

1. Nossa escolha de 10 para o K foi arbitrária — que efeito diferentes valores de K têm nos resultados?

2. Nossa métrica de distância também foi um tanto arbitrária — apenas pegamos a distância de cosseno entre os gêneros e somamos com a diferença entre as pontuações de popularidade normalizadas. Você consegue melhorar isso?

In [2]:
# Mesma preparação da aula: carrega os dados, calcula popularidade normalizada e monta o movieDict
import pandas as pd
import numpy as np
from scipy import spatial
import operator

r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('C:/Users/zinho/OneDrive/Documentos/Lamia/Card 11 - Prática Lidando com Dados do Mundo Real (II)/Aquivos_de_C%C3%B3digo/u.data', sep='\t', names=r_cols, usecols=range(3))

movieProperties = ratings.groupby('movie_id').agg({'rating': ['size', 'mean']})

movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)))

movieDict = {}
with open('C:/Users/zinho/OneDrive/Documentos/Lamia/Card 11 - Prática Lidando com Dados do Mundo Real (II)/Aquivos_de_C%C3%B3digo/u.item') as f:
    for line in f:
        fields = line.rstrip('\n').split('|')
        movieID = int(fields[0])
        name = fields[1]
        genres = list(map(int, fields[5:25]))
        # guardo a média separada pra facilitar as contas depois
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID]['rating']['mean'])

In [3]:
# Métrica original da aula: cosseno dos gêneros + diferença de popularidade
def ComputeDistance(a, b):
    genreDistance = spatial.distance.cosine(a[1], b[1])
    popularityDistance = abs(a[2] - b[2])
    return genreDistance + popularityDistance

# Mesmo getNeighbors da aula, só que agora aceitando qualquer função de distância
def getNeighbors(movieID, K, distFunc=ComputeDistance):
    distances = []
    for movie in movieDict:
        if movie != movieID:
            dist = distFunc(movieDict[movieID], movieDict[movie])
            distances.append((movie, dist))
    distances.sort(key=operator.itemgetter(1))
    return [distances[x][0] for x in range(K)]

**Parte 1 — Testando diferentes valores de K**

In [4]:
testMovie = 1  # Toy Story
realRating = movieDict[testMovie][3]
print(f'Nota real de {movieDict[testMovie][0]}: {realRating:.4f}\n')

for K in [1, 3, 5, 10, 15, 25, 50, 100]:
    neighbors = getNeighbors(testMovie, K)
    # a previsão é só a média das notas médias dos vizinhos
    avgRating = np.mean([movieDict[n][3] for n in neighbors])
    erro = abs(avgRating - realRating)
    print(f'K = {K:3d} -> previsão: {avgRating:.4f} | erro: {erro:.4f}')

Nota real de Toy Story (1995): 3.8783

K =   1 -> previsão: 3.1567 | erro: 0.7216
K =   3 -> previsão: 3.5338 | erro: 0.3445
K =   5 -> previsão: 3.7190 | erro: 0.1594
K =  10 -> previsão: 3.3446 | erro: 0.5337
K =  15 -> previsão: 3.4669 | erro: 0.4114
K =  25 -> previsão: 3.5086 | erro: 0.3697
K =  50 -> previsão: 3.3875 | erro: 0.4908
K = 100 -> previsão: 3.3033 | erro: 0.5750


**Observação:** com K muito pequeno a previsão fica sensível demais a um vizinho estranho, e com K muito grande entram filmes cada vez menos parecidos, puxando a previsão pra média geral. O melhor resultado costuma ficar num meio-termo — vale olhar qual K deu o menor erro na saída acima.

**Parte 2 — Melhorando a métrica de distância**

In [5]:
# A nova metrica adiciona a diferença de nota média (normalizada pra ficar entre 0 e 1,
# já que as notas vão de 1 a 5) e dá pesos diferentes pra cada componente,
# priorizando o genero, que eh o que mais define se dois filmes são parecidos.
def ComputeDistanceV2(a, b, wGenre=0.6, wPop=0.2, wRating=0.2):
    genreDistance = spatial.distance.cosine(a[1], b[1])
    popularityDistance = abs(a[2] - b[2])
    ratingDistance = abs(a[3] - b[3]) / 4.0  # 4 é a diferença máxima possível (5 - 1)
    return wGenre * genreDistance + wPop * popularityDistance + wRating * ratingDistance

In [6]:
# Comparando as duas métricas com o mesmo K
K = 10

print('--- Vizinhos com a métrica original ---')
neighbors = getNeighbors(testMovie, K, ComputeDistance)
for n in neighbors:
    print(f'{movieDict[n][0]} (média: {movieDict[n][3]:.2f})')
predOld = np.mean([movieDict[n][3] for n in neighbors])

print('\n--- Vizinhos com a métrica melhorada ---')
neighbors = getNeighbors(testMovie, K, ComputeDistanceV2)
for n in neighbors:
    print(f'{movieDict[n][0]} (média: {movieDict[n][3]:.2f})')
predNew = np.mean([movieDict[n][3] for n in neighbors])

print(f'\nNota real:            {realRating:.4f}')
print(f'Previsão (original):  {predOld:.4f} | erro: {abs(predOld - realRating):.4f}')
print(f'Previsão (melhorada): {predNew:.4f} | erro: {abs(predNew - realRating):.4f}')

--- Vizinhos com a métrica original ---
Liar Liar (1997) (média: 3.16)
Aladdin (1992) (média: 3.81)
Willy Wonka and the Chocolate Factory (1971) (média: 3.63)
Monty Python and the Holy Grail (1974) (média: 4.07)
Full Monty, The (1997) (média: 3.93)
George of the Jungle (1997) (média: 2.69)
Beavis and Butt-head Do America (1996) (média: 2.79)
Birdcage, The (1996) (média: 3.44)
Home Alone (1990) (média: 3.09)
Aladdin and the King of Thieves (1996) (média: 2.85)

--- Vizinhos com a métrica melhorada ---
Aladdin (1992) (média: 3.81)
Aladdin and the King of Thieves (1996) (média: 2.85)
Pinocchio (1940) (média: 3.67)
Winnie the Pooh and the Blustery Day (1968) (média: 3.80)
Grand Day Out, A (1992) (média: 4.11)
Wrong Trousers, The (1993) (média: 4.47)
Willy Wonka and the Chocolate Factory (1971) (média: 3.63)
Home Alone (1990) (média: 3.09)
Sword in the Stone, The (1963) (média: 3.33)
Matilda (1996) (média: 3.21)

Nota real:            3.8783
Previsão (original):  3.3446 | erro: 0.5337
Previ

**Observação:** a métrica nova tende a trazer vizinhos com nota mais próxima da do Toy Story (porque agora a nota entra na conta) e mantém o gênero como fator principal. O resultado é uma previsão com erro menor. Dá pra ir além ajustando os pesos, testando outras distâncias pros gêneros (Jaccard, por exemplo) ou incluindo mais atributos do u.item.